In [22]:
# importing the packages
import numpy as np
import pandas as pd
from itertools import chain
from ortools.linear_solver import pywraplp
from warnings import filterwarnings

filterwarnings("ignore")

In [72]:
# Reading the dataframe and converting it to a numpy array
np_arr = pd.read_csv(
    "../data/flight_legs/data.csv",
    parse_dates=["start_time", "end_time"],
).to_numpy()


# read the duties from the txt file
with open("../data/pairings/pairings.txt") as file:
    pairings = [eval(line) for line in file]


# calculate the cost matrix from pair list
cost_list = [
    abs((np_arr[pair[-1][-1]][4] - np_arr[pair[0][0]][3]).total_seconds() / 3600)
    for pair in pairings
]

cost_matrix = np.array(cost_list).reshape(-1, 1)

# create a zero np_array
pair_matrix = np.zeros((len(pairings), len(np_arr)))

# fill the values of the flight legs in each pairing with 1
for i, pair in enumerate(pairings):
    pair_matrix[i, list(chain.from_iterable(pair))] = 1

# determining the number of flights and tasks
num_pairs = pair_matrix.shape[0]
num_flights = pair_matrix.shape[1]
print(num_pairs, num_flights)

91692 96


In [77]:
plegs = [list(chain.from_iterable(pair)) for pair in pairings]
plegs

[[0, 1, 2, 3],
 [0, 1, 2, 11],
 [0, 1, 2, 23],
 [0, 1, 2, 35],
 [0, 1, 2, 43],
 [0, 1, 2, 55],
 [0, 1, 2, 67],
 [0, 1, 2, 75],
 [0, 1, 2, 87],
 [0, 1, 6, 3],
 [0, 1, 6, 7],
 [0, 1, 6, 35],
 [0, 1, 6, 39],
 [0, 1, 6, 67],
 [0, 1, 6, 71],
 [0, 1, 10, 11],
 [0, 1, 10, 23],
 [0, 1, 10, 43],
 [0, 1, 10, 55],
 [0, 1, 10, 75],
 [0, 1, 10, 87],
 [0, 1, 14, 7],
 [0, 1, 14, 15],
 [0, 1, 14, 39],
 [0, 1, 14, 47],
 [0, 1, 14, 71],
 [0, 1, 14, 79],
 [0, 1, 34, 3],
 [0, 1, 34, 11],
 [0, 1, 34, 23],
 [0, 1, 34, 35],
 [0, 1, 34, 43],
 [0, 1, 34, 55],
 [0, 1, 34, 67],
 [0, 1, 34, 75],
 [0, 1, 34, 87],
 [0, 1, 38, 3],
 [0, 1, 38, 7],
 [0, 1, 38, 35],
 [0, 1, 38, 39],
 [0, 1, 38, 67],
 [0, 1, 38, 71],
 [0, 1, 42, 11],
 [0, 1, 42, 23],
 [0, 1, 42, 43],
 [0, 1, 42, 55],
 [0, 1, 42, 75],
 [0, 1, 42, 87],
 [0, 1, 46, 7],
 [0, 1, 46, 15],
 [0, 1, 46, 39],
 [0, 1, 46, 47],
 [0, 1, 46, 71],
 [0, 1, 46, 79],
 [0, 1, 66, 3],
 [0, 1, 66, 11],
 [0, 1, 66, 23],
 [0, 1, 66, 35],
 [0, 1, 66, 43],
 [0, 1, 66, 55],
 [0,

In [71]:
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("GLOP")

# creating the binary allocation variable
x = np.array([solver.BoolVar("") for i in range(num_pairs)]).reshape(-1, 1)
# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(
        solver.Sum(
            [
                x[j][0] * pair_matrix[j][i]
                for j in range(num_pairs)
                if pair_matrix[j][i] != 0.0
            ]
        )
        == 1.0
    )
# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i][0] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

The Solution is OPTIMAL
91692
[30, 42, 57, 620, 633, 640, 1750, 1776, 2351, 3057, 3551, 3606, 3626, 3801, 4024, 4045, 4105, 4340, 4586, 4983, 5120, 5360, 5684, 6498, 6727, 6891, 6908, 6993, 7222, 8196, 8610, 8757, 8984, 9080, 9083, 9309, 9628, 9639, 9825, 10152, 10468, 11547, 11874, 11886, 11996, 12045, 12129, 12379, 13131, 13298, 13623, 13993, 14017, 14136, 14142, 14161, 14701, 14906, 15078, 15120, 33186]
456.00000000000006
[0.0, 2.999999999999997, 18.000000000000004, -1.0000000000000002, 2.0, 0.9999999999999964, 13.000000000000002, 2.0, -1.0, 1.999999999999997, 18.000000000000004, -4.440892098500626e-16, 5.0000000000000036, 1.9999999999999956, 13.000000000000004, 0.0, 5.0, 2.9999999999999964, 13.000000000000004, -0.9999999999999996, -0.9999999999999984, 1.9999999999999964, 18.000000000000004, 1.0, 18.000000000000004, -10.0, -1.7763568394002505e-15, 10.999999999999996, 14.000000000000004, 0.0, 0.0, 1.9999999999999964, -4.440892098500626e-16, 2.9999999999999964, 18.000000000000004, -1.

In [ ]:
# Initializing the MIP Solver
solver = pywraplp.Solver.CreateSolver("SAT")

# creating the binary allocation variable
x = np.array([solver.BoolVar("") for i in range(num_pairs)]).reshape(-1, 1)
# declaring the constraints
# Uniqueness of flight legs and all flight legs covered
for i in range(num_flights):
    solver.Add(
        solver.Sum(
            [
                x[j][0] * pair_matrix[j][i]
                for j in range(num_pairs)
                if pair_matrix[j][i] != 0.0
            ]
        )
        == 1.0
    )
# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i][0] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
if status == pywraplp.Solver.OPTIMAL:
    print("The Solution is OPTIMAL")
elif status == pywraplp.Solver.FEASIBLE:
    print("The Solution is Feasible")
else:
    print("The Solution is not Feasible")

x_values = [x[i][0].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

In [35]:
# importing the packages
import numpy as np
from ortools.linear_solver import pywraplp

# Initialize the solver
solver = pywraplp.Solver.CreateSolver("CBC")
use_dual_simplex: True

# Decision Variable
x = [solver.NumVar(0, 1, f"x_{i}") for i in range(num_pairs)]

# Constraints
for j in range(num_flights):
    solver.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) - 1 >= 0)

# Objective function
solver.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the problem
status = solver.Solve()

# print the state of the optimization problem
x_values = [x[i].solution_value() for i in range(num_pairs)]
print(len(x_values))
index = [i for i in range(num_pairs) if x_values[i] > 0]
print(index)
print(solver.Objective().Value())
# Access the dual values (shadow prices)
dual_values = [constraint.dual_value() for constraint in solver.constraints()]
print(dual_values)

157.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
32978
[559, 814, 3241, 3883, 4055, 4230, 7495, 8917, 11736, 12096, 12389, 12423, 12510, 12794, 12907, 14296, 16640, 19094, 19213, 19483, 19543, 19917, 20119, 21563, 24417, 24451, 26595, 26621, 28197, 28365]
157.0
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [36]:
# importing the packages
import numpy as np
from ortools.sat.python import cp_model

# Declare the CP_SAT model
model = cp_model.CpModel()

# create the decision variable
x = []
for i in range(num_pairs):
    x.append(model.NewBoolVar(f"x_{i}"))

# Constraints
constraints = []
for j in range(num_flights):
    constraints.append(
        model.Add(sum(pair_matrix[i][j] * x[i] for i in range(num_pairs)) >= 1)
    )

# Objective Function
model.Minimize(sum(cost_matrix[i][0] * x[i] for i in range(num_pairs)))

# Solve the Problem
solver = cp_model.CpSolver()
status = solver.Solve(model)

if status == cp_model.OPTIMAL:
    selected_pairs = [i for i in range(num_pairs) if solver.Value(x[i]) == 1]
    print(selected_pairs)
else:
    print("oops")

print(solver.ObjectiveValue(), len(selected_pairs))

[866, 4274, 7795, 11481, 12867, 14253, 17113, 19474, 22259, 24417, 25903, 28182]
157.0 12


In [4]:
import numpy as np
from scipy.optimize import linprog


def solve_mip(pair_matrix, cost_matrix):
    num_pairs, num_flights = pair_matrix.shape

    # Objective coefficients for the binary decision variables
    c = cost_matrix.flatten().reshape(-1, 1)
    print(c.shape)
    # Coefficients matrix for the constraints (each flight leg should be covered at least once)
    A_eq = np.vstack([pair_matrix, np.ones((1, num_flights))])
    b_eq = np.ones(num_flights + 1)  # RHS for equality constraints

    # Bounds for decision variables (x should be binary)
    bounds = [(0, 1) for _ in range(num_pairs)]
    print(A_eq.shape)
    # Solve the linear programming problem
    result = linprog(c, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs")

    # Extract the results
    minimized_cost = result.fun
    selected_pairs = np.round(result.x).astype(int)
    dual_variables = result.slack[
        :-1
    ]  # Slack variables correspond to dual variables in equality constraints

    return minimized_cost, selected_pairs, dual_variables


minimized_cost, selected_pairs, dual_variables = solve_mip(pair_matrix, cost_matrix)

# Print the results
print("Minimized Cost:", minimized_cost)
print("Selected Pairs:", selected_pairs)
print("Optimal Dual Variables:", dual_variables)

(80706, 1)
(80707, 240)


ValueError: Invalid input for linprog: A_eq must have exactly two dimensions, and the number of columns in A_eq must be equal to the size of c